In [ ]:
!unzip /content/drive/MyDrive/amazonreview.zip -d /content/data

Archive:  /content/drive/MyDrive/amazonreview.zip
  inflating: /content/data/7817_1.csv  


In [ ]:
import pandas as pd

df = pd.read_csv('/content/data/7817_1.csv')
df.head()

,id,asins,brand,categories,colors,dateAdded,dateUpdated,dimension,ean,keys,...,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username,sizes,upc,weight
0,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I initially had trouble deciding between the p...,"Paperwhite voyage, no regrets!",NaN,NaN,Cristina M,NaN,NaN,205 grams
1,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,Allow me to preface this with a little history...,One Simply Could Not Ask For More,NaN,NaN,Ricky,NaN,NaN,205 grams
2,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,4.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I am enjoying it so far. Great for reading. Ha...,Great for those that just want an e-reader,NaN,NaN,Tedd Gardiner,NaN,NaN,205 grams
3,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I bought one of the first Paperwhites and have...,Love / Hate relationship,NaN,NaN,Dougal,NaN,NaN,205 grams
4,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I have to say upfront - I don't like coroporat...,I LOVE IT,NaN,NaN,Miljan David Tanic,NaN,NaN,205 grams


In [ ]:
def classify_review(text):
    text = text.lower()

    quality_keywords = [
        "broken","defect","defective","poor quality","cheap quality","low quality",
        "battery","battery issue","battery drain","battery dies","not working",
        "stopped working","malfunction","malfunctioning","damage","damaged",
        "cracked","screen cracked","faulty","heating issue","overheating",
        "doesn't work","does not work","hardware issue","software bug",
        "lagging","slow performance","freezing","glitch","error","problem",
        "bad build","loose","weak material","poor build","not durable"
    ]

    delivery_keywords = [
        "delivery","shipping","shipment","courier","arrived late","late delivery",
        "delayed","delay","delivery delay","slow delivery","package delay",
        "tracking issue","wrong address","delivered late","not delivered",
        "missing package","lost package","damaged package","shipping problem",
        "delivery problem","delivery issue","courier issue","wrong item delivered",
        "incomplete order","partial delivery"
    ]

    feature_keywords = [
        "should have","should include","would like","wish it had","wish it came with",
        "missing feature","needs improvement","could be better","lack of feature",
        "needs update","add feature","please add","feature request",
        "it would be better if","i would prefer","improvement needed",
        "needs better","needs upgrade","future version should",
        "hope they add","should improve"
    ]

    positive_keywords = [
        "great","excellent","amazing","awesome","perfect","very good",
        "high quality","fantastic","love it","loved it","best product",
        "worth it","value for money","satisfied","happy with","works perfectly",
        "good quality","superb","brilliant","impressed","recommended",
        "five stars","good purchase","very satisfied","nice product"
    ]

    if any(word in text for word in quality_keywords):
        return 0   # Quality complaint

    elif any(word in text for word in delivery_keywords):
        return 1   # Delivery complaint

    elif any(word in text for word in feature_keywords):
        return 3   # Feature request

    elif any(word in text for word in positive_keywords):
        return 2   # Positive feedback

    else:
        return 2   # Default to positive/neutral

In [ ]:
df = df[['reviews.text']]
df = df.dropna()
df.head()

,reviews.text
0,I initially had trouble deciding between the p...
1,Allow me to preface this with a little history...
2,I am enjoying it so far. Great for reading. Ha...
3,I bought one of the first Paperwhites and have...
4,I have to say upfront - I don't like coroporat...


In [ ]:
df = df.rename(columns={'reviews.text': 'review'})

In [ ]:
df['label'] = df['review'].apply(classify_review)

In [ ]:
df['label'].value_counts()

,count
label,
2,1203
0,343
3,41
1,10


In [ ]:
max_samples = 300

df_balanced = df.groupby('label').apply(
    lambda x: x.sample(min(len(x), max_samples), random_state=42)
).reset_index(drop=True)

/tmp/ipykernel_326/2735862224.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanced = df.groupby('label').apply(


In [ ]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df_balanced['review'],
    df_balanced['label'],
    test_size=0.2,
    random_state=42
)

In [ ]:
!pip install transformers
!pip install torch

In [ ]:
from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=128
)

In [ ]:
import torch

class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = ReviewDataset(train_encodings, train_labels)
test_dataset = ReviewDataset(test_encodings, test_labels)

In [ ]:
import torch.nn as nn
from transformers import RobertaModel

In [ ]:
class RobertaGRU(nn.Module):
    def __init__(self, num_classes):
        super(RobertaGRU, self).__init__()

        self.roberta = RobertaModel.from_pretrained("roberta-base")

        self.gru = nn.GRU(
            input_size=768,
            hidden_size=256,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(512, num_classes)

    def forward(self, input_ids, attention_mask):

        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        sequence_output = outputs.last_hidden_state

        gru_output, _ = self.gru(sequence_output)

        final_output = gru_output[:, -1, :]

        out = self.dropout(final_output)

        logits = self.fc(out)

        return logits

In [ ]:
model = RobertaGRU(num_classes=4)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

In [ ]:
for epoch in range(5):

    model.train()

    for batch in train_loader:

        optimizer.zero_grad()

        outputs = model(p
            batch['input_ids'],
            batch['attention_mask']
        )

        loss = criterion(outputs, batch['labels'])

        loss.backward()

        optimizer.step()

    print("Epoch completed")

Epoch completed
Epoch completed
Epoch completed
Epoch completed
Epoch completed


In [ ]:
model.eval()

predictions = []
true_labels = []

with torch.no_grad():

    for batch in DataLoader(test_dataset, batch_size=16):

        outputs = model(
            batch['input_ids'],
            batch['attention_mask']
        )

        preds = torch.argmax(outputs, dim=1)

        predictions.extend(preds.tolist())
        true_labels.extend(batch['labels'].tolist())

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(true_labels, predictions))

              precision    recall  f1-score   support

           0       0.82      0.93      0.87        71
           1       0.00      0.00      0.00         2
           2       0.84      0.80      0.82        51
           3       1.00      0.29      0.44         7

    accuracy                           0.83       131
   macro avg       0.67      0.50      0.53       131
weighted avg       0.83      0.83      0.82       131



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
